# Módulo 3 · Clase 3: No Linealidad — Árboles de Decisión y Random Forest

**Machine Learning for Petroleum Engineers Using Python**  
SLB Ecuador · UDLA · 2026  

Instructor: **Carlos Enrique Mosquera Trujillo**  
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## La idea de hoy

La regresión lineal y la logística comparten un límite: **por dentro son una recta** — asumen que el paso es *parejo* (el mismo cambio en X produce siempre el mismo cambio en Y). Pero la física de producción está llena de **curvas, mesetas, umbrales e interacciones**: el choke, el declive, la curva IPR…

Hoy: (1) **ver** a la recta fracasar, (2) construir **árboles de decisión** desde cero — la versión automatizada de los cutoffs que ya usan —, (3) juntarlos en un **Random Forest**, y (4) **batir el récord de costo de la Clase 2**.

> 💡 Las celdas 🧩 de práctica están **en blanco**: te toca escribirlas.

---
# 0 · Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, r2_score, mean_absolute_error)

URL_LITO = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/litologia_force2020.csv"
URL_OPER = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/operacion_pozos_volve.csv"
print("listo")

---
# 1 · Ver el límite de la recta: las dos lunas

`make_moons` genera dos grupos con forma de **medialunas entrelazadas**. Cualquier persona los separa a ojo. ¿Puede separarlos una **frontera recta**?

In [ ]:
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=600, noise=0.22, random_state=7)

plt.scatter(X_m[y_m==0,0], X_m[y_m==0,1], s=10, color='gray', label='clase A')
plt.scatter(X_m[y_m==1,0], X_m[y_m==1,1], s=10, color='orange', label='clase B')
plt.legend(); plt.title('Las dos lunas')

In [ ]:
# funcion para dibujar la frontera de decision de cualquier modelo
def frontera(modelo, titulo):
    xx, yy = np.meshgrid(np.linspace(-1.7, 2.7, 300), np.linspace(-1.2, 1.7, 300))
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.2, levels=1, colors=['gray','orange'])
    plt.scatter(X_m[y_m==0,0], X_m[y_m==0,1], s=10, color='gray')
    plt.scatter(X_m[y_m==1,0], X_m[y_m==1,1], s=10, color='orange')
    plt.title(titulo)

log_m = LogisticRegression().fit(X_m, y_m)
frontera(log_m, f'Logistica: acc={log_m.score(X_m, y_m):.2f}  (no puede doblar)')

La logística se queda en **0.86 para siempre**: no es falta de datos, es un límite **estructural** — su frontera es una recta. Guarda esta función `frontera`: la usaremos de nuevo cuando tengamos árboles.

---
# 2 · Problema 1 (clasificación): batir el récord de la Clase 2

## Acotar el problema (siempre, antes de tocar datos)

| | |
|---|---|
| **Pregunta de negocio** | ¿qué intervalos del pozo son **reservorio** (arenisca) y cuáles **sello** (lutita)? |
| **Target** | `LITH` → binarizado: arenisca = 1 (la clase que queremos cazar) |
| **Features** | los 5 registros: `GR`, `RHOB`, `NPHI`, `DTC`, `RDEP` |
| **Métrica de éxito** | el **costo de negocio** de la Clase 2: FN (zona perdida) = 10 × FP (cañoneo en seco) |
| **Récord a batir** | logística: accuracy 0.944 · recall 0.79 · **costo 5 953** |

¿Por qué el mismo dataset? Para una comparación **justa**: mismos datos, mismo test, mismas métricas. **Solo cambia el modelo.**

In [ ]:
lito = pd.read_csv(URL_LITO)
lito["es_reservorio"] = (lito["LITH"] == "Sandstone").astype(int)   # binarizar (Clase 2)
print(lito.shape)
lito.head(3)

## Exploración (repaso rápido, con ojos nuevos)

Ya conocemos este dataset — pero ahora buscamos **evidencia de no linealidad**: umbrales y traslapes.

In [ ]:
# el desbalance que lo define todo
lito["LITH"].value_counts(normalize=True).round(3)

In [ ]:
# medias por clase: ¿que variable separa mas?
feats = ["GR", "RHOB", "NPHI", "DTC", "RDEP"]
lito.groupby("LITH")[feats].mean().round(2)

In [ ]:
# el traslape: la razon por la que un solo cutoff no basta
arena = lito[lito["es_reservorio"] == 1]
lutita = lito[lito["es_reservorio"] == 0]
ax = lutita["GR"].plot(kind="hist", bins=60, alpha=0.6, label="lutita")
arena["GR"].plot(kind="hist", bins=60, alpha=0.6, ax=ax, label="arenisca")
ax.set_xlim(0, 200); ax.legend(); ax.set_xlabel("GR (API)")
ax.set_title("Separan... pero se traslapan: territorio de arboles")

In [ ]:
# split estratificado (mismo que la Clase 2, para comparar contra el record)
X = lito[feats]
y = lito["es_reservorio"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)
print(len(X_tr), len(X_te))

## La línea base: reconstruir el récord de la Clase 2

In [ ]:
sc = StandardScaler()
logi = LogisticRegression(max_iter=1000).fit(sc.fit_transform(X_tr), y_tr)
pred_logi = logi.predict(sc.transform(X_te))

def reporte(y_real, pred, nombre):
    tn, fp, fn, tp = confusion_matrix(y_real, pred).ravel()
    print(f'{nombre:22} acc={accuracy_score(y_real, pred):.3f} '
          f'rec={recall_score(y_real, pred):.3f} prec={precision_score(y_real, pred):.3f} '
          f'FN={fn:4d} FP={fp:4d} costo={fn*10 + fp}')

reporte(y_te, pred_logi, "Logistica (Clase 2)")

## El árbol de decisión, paso a paso

Un árbol es una **cadena de preguntas de sí/no** — la versión automatizada de los cutoffs del intérprete (*"si GR < 60 → arena"*). La diferencia: la máquina **prueba todos los cortes posibles de todas las variables** y se queda con los que dejan los grupos más **puros** (casi todos de la misma clase).

### Empecemos con UNA sola pregunta

In [ ]:
# profundidad 1 sobre una sola variable: ¿que corte de GR elige la maquina?
t1 = DecisionTreeClassifier(max_depth=1)
t1.fit(lito[["GR"]], y)

print('corte elegido: GR =', round(t1.tree_.threshold[0], 1))
print('accuracy con UNA pregunta:', round(t1.score(lito[["GR"]], y), 3))
print('(el modelo tonto daba 0.828)')

**GR = 49** — casi el mismo cutoff que un intérprete usaría por experiencia (~50–60). Y con **una sola pregunta** ya le gana al modelo tonto.

### Dejar que encadene preguntas: el árbol crece

In [ ]:
arbol2 = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)

plt.figure(figsize=(12, 5))
plot_tree(arbol2, feature_names=feats, class_names=['lutita','arenisca'],
          filled=True, rounded=True, fontsize=9, impurity=False, proportion=True)
plt.show()

Léelo de arriba hacia abajo: la **raíz** pregunta por GR (la misma pregunta de antes) y luego **refina con NPHI** en ambos lados, con cortes distintos en cada lado. GR bajo + NPHI bajo → arenisca. La física, aprendida de los datos.

### Ahora sí: las lunas con un árbol

In [ ]:
arbol_m = DecisionTreeClassifier(max_depth=8, random_state=0).fit(X_m, y_m)
frontera(arbol_m, f'Arbol: acc={arbol_m.score(X_m, y_m):.2f}  (escalones que siguen la forma)')

### El árbol en nuestro problema real

In [ ]:
arbol5 = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_tr, y_tr)
reporte(y_te, arbol5.predict(X_te), "Arbol prof 5")
reporte(y_te, pred_logi, "Logistica (Clase 2)")

Con 5 niveles de preguntas ya supera a la logística — **sin estandarizar** (los árboles comparan cada variable consigo misma; las escalas no importan).

### La tentación: crecer sin límite → overfitting

In [ ]:
profundidades = range(1, 21)
acc_tr, acc_te = [], []
for p in profundidades:
    t = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_tr, y_tr)
    acc_tr.append(t.score(X_tr, y_tr))
    acc_te.append(t.score(X_te, y_te))

plt.plot(profundidades, acc_tr, marker='o', label='train (memoriza)')
plt.plot(profundidades, acc_te, marker='o', label='test (aprende)')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend()
plt.title('Memoriza mas; aprende hasta cierto punto')

Sin límite, el árbol llega a accuracy **1.000 en train** (una hoja para cada rincón de los datos) pero en test deja de mejorar: aprendió el **ruido**. Es el overfitting de la Clase 1, ahora **visible en una curva**. Además el árbol solo es **inestable**: cambien unos datos y puede elegir otra primera pregunta — y cambiar entero.

## Random Forest: la sabiduría de la multitud

La solución no es un árbol mejor: es **dejar de confiar en uno**. `RandomForest` entrena **cientos de árboles**, cada uno:

- con una **muestra al azar** de los datos (como geólogos que estudiaron pozos distintos), y
- mirando un **subconjunto al azar de variables** en cada corte (para que no todos se obsesionen con el GR).

Cada árbol **vota**; el porcentaje de votos funciona como **probabilidad** → el umbral y la matriz de costos de la Clase 2 siguen valiendo. Los errores individuales (cada árbol memoriza ruido *distinto*) **se cancelan al votar**; queda la señal.

In [ ]:
bosque = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
bosque.fit(X_tr, y_tr)

reporte(y_te, pred_logi, "Logistica (Clase 2)")
reporte(y_te, arbol5.predict(X_te), "Arbol prof 5")
reporte(y_te, bosque.predict(X_te), "Random Forest")

**El marcador:** el bosque mejora las cuatro celdas de la matriz a la vez — recall de 0.79 a **0.93** (de perder 2 de cada 10 zonas a menos de 1) y el costo cae de 5 953 a **2 158**.

### ¿En qué se fija el bosque?

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=feats)
importancias.sort_values().plot(kind='barh', title='Importancia de variables')

`GR` domina (37 %) — coherente con que el mejor corte univariado era GR=49 — y `NPHI` es la segunda, la misma que el árbol pequeño eligió para refinar. **Ojo:** importancia ≠ causalidad; dice qué *usó* el modelo, no qué *hace* que la roca sea arena.

### El remate: la perilla de la Clase 2 sobre el bosque

In [ ]:
proba = bosque.predict_proba(X_te)[:, 1]

umbrales = np.linspace(0.05, 0.95, 60)
costos = []
for u in umbrales:
    tn, fp, fn, tp = confusion_matrix(y_te, (proba >= u).astype(int)).ravel()
    costos.append(fn*10 + fp)

mejor = umbrales[int(np.argmin(costos))]
plt.plot(umbrales, costos)
plt.axvline(0.5, ls='--', color='gray', label='0.5 por defecto')
plt.axvline(mejor, color='green', label=f'optimo ({mejor:.2f})')
plt.xlabel('umbral'); plt.ylabel('costo total'); plt.legend()
plt.title('El umbral que minimiza el costo')

In [ ]:
pred_final = (proba >= mejor).astype(int)
reporte(y_te, pred_final, f"RF + umbral {mejor:.2f}")

**De 5 953 (logística) a ~1 150: 81 % menos costo.** Solo 42 zonas perdidas donde antes eran 563. Las dos clases se **suman**: mejor modelo (hoy) × mejor decisión (Clase 2).

**Cómo reportarlo al gerente** (regla de la Clase 2): no *"el F1 subió a 0.94"* sino *"dejamos de perder 521 zonas productivas, a cambio de ~400 pruebas adicionales que salen secas — 81 % menos costo total"*.

---
## 🧩 Práctica 1: Tu propio bosque

1. Entrena un `DecisionTreeClassifier` con `max_depth=3` y grafícalo con `plot_tree`. ¿Qué preguntas eligió? ¿Coinciden con la física?
2. Compara accuracy **train vs test** para `max_depth` = 2, 5, 10 y sin límite. ¿Dónde empieza a memorizar?
3. Entrena un `RandomForestClassifier` con `n_estimators=50` y con `500`. ¿Cambia mucho el resultado? ¿Qué concluyes?
4. El equipo de yacimientos dice que en este campo FN = **5**×FP (no 10×). Recalcula el umbral óptimo. ¿Sube o baja? ¿Por qué?

In [ ]:
# Escribe tu solucion aqui


---
# 3 · Problema 2 (regresión): un medidor virtual de flujo

Los árboles y bosques también predicen **números**. Caso real para demostrarlo — y para ver a la regresión lineal quedarse corta.

## Acotar el problema

| | |
|---|---|
| **Pregunta de negocio** | ¿cuánto está produciendo cada pozo **hoy**, sin desviar el flujo al separador de prueba? |
| **Target** | `oil` — producción diaria medida (Sm³/día) |
| **Features** | los sensores siempre disponibles: `p_fondo`, `p_cabeza`, `t_cabeza`, `choke`, `dp_choke`, `horas` |
| **Métrica de éxito** | MAE (error típico en Sm³/día) y R², en test |
| **Por qué importa** | los *virtual flow meters* son un producto real: la prueba de pozo es esporádica; los sensores, continuos |

## Las variables, una por una

| Columna | Qué mide | Dónde |
|---------|----------|-------|
| `p_fondo` | presión en el fondo del pozo (bar) | sensor de fondo |
| `p_cabeza` | presión en la cabeza del pozo (bar) | superficie |
| `t_cabeza` | temperatura en cabeza (°C) | superficie |
| `choke` | apertura de la válvula choke (%) | superficie |
| `dp_choke` | caída de presión a través del choke (bar) | superficie |
| `horas` | horas en operación ese día | registro operativo |
| **`oil`** | **el target**: producción de petróleo (Sm³/día) | separador |

In [ ]:
op = pd.read_csv(URL_OPER)
print(op.shape, '|', op['pozo'].nunique(), 'pozos')
op.head(3)

## Exploración: buscar la no linealidad

In [ ]:
op.describe().round(1)

In [ ]:
# correlaciones con el target
feats_op = ["horas", "p_fondo", "p_cabeza", "t_cabeza", "choke", "dp_choke"]
op[feats_op].corrwith(op["oil"]).round(2).sort_values()

In [ ]:
# la relacion mas fuerte, dibujada: tiene forma de codo
op.plot(kind="scatter", x="p_cabeza", y="oil", s=3, alpha=0.25,
        title="La relacion existe... pero NO es una recta")

El **codo** es la firma de la no linealidad: a presiones bajas la producción casi no responde; en la zona media se dispara. Una recta está obligada a pasar *por en medio de nada*.

## Línea base lineal vs bosque

In [ ]:
Xo = op[feats_op]
yo = op["oil"]
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(Xo, yo, test_size=0.25, random_state=42)

lineal = LinearRegression().fit(Xo_tr, yo_tr)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xo_tr, yo_tr)

for nombre, m in [('Lineal', lineal), ('Random Forest', rf_reg)]:
    p = m.predict(Xo_te)
    print(f'{nombre:15} R2={r2_score(yo_te, p):.3f}  MAE={mean_absolute_error(yo_te, p):.0f} Sm3/dia')

**De 583 a 78 Sm³/día de error típico: 7.5 veces menos.** Para un pozo de 1 264 Sm³/día de media, es pasar de inutilizable a operativo.

### ¿Qué sensor domina?

In [ ]:
pd.Series(rf_reg.feature_importances_, index=feats_op).sort_values().plot(
    kind='barh', title='Importancias del medidor virtual')

In [ ]:
# el veredicto visual
p_rf = rf_reg.predict(Xo_te)
comp = pd.DataFrame({"real": yo_te, "predicho": p_rf})
ax = comp.plot(kind="scatter", x="real", y="predicho", s=4, alpha=0.3)
lims = [0, float(max(yo_te.max(), p_rf.max()))]
ax.plot(lims, lims, color='red', label='prediccion perfecta')
ax.legend(); ax.set_title(f'Predicho vs real (R2={r2_score(yo_te, p_rf):.2f})')

`p_cabeza` (49 %) y `t_cabeza` (33 %) dominan — coincide con la física: lo que pasa por el choke se refleja en las condiciones de cabeza. Y a diferencia de la recta del declive (Clase 1), el bosque **nunca predice producción negativa**: no inventa fuera del rango que vio.

---
## 🧩 Práctica 2: Afinar el medidor virtual

1. Entrena un `DecisionTreeRegressor` con `max_depth=5` y sin límite. Compara R² train vs test: ¿se repite el patrón de overfitting?
2. Con el bosque, calcula el MAE **por pozo** (agrupa el test por `pozo`). ¿En cuál funciona peor? ¿Por qué crees?
3. Quita `p_cabeza` de las features y reentrena el bosque. ¿Cuánto empeora? ¿Qué te dice eso sobre la dependencia de un solo sensor?
4. Escribe una frase para el superintendente: ¿recomendarías usar este medidor virtual mientras el separador está ocupado? ¿Con qué margen de error debe contar?

In [ ]:
# Escribe tu solucion aqui


---
## Cierre

**El flujo actualizado:** `datos` → `train/test` → `línea base explicable` → `modelo no lineal` → `métricas + costo` → `umbral por negocio`

**Conceptos:** no linealidad (el paso deja de ser parejo) · la recta fracasa (lunas) · árbol = preguntas encadenadas (raíz/hojas/profundidad) · cómo la máquina elige el corte (pureza) · overfitting visible (train 1.000) · Random Forest (diversidad + voto) · importancias ≠ causalidad · poder vs explicabilidad.

**El número del día:** costo de la litología: logística **5 953** → bosque + umbral **1 150** (81 % menos). Mejor modelo × mejor decisión.

**Herramientas:** `DecisionTreeClassifier/Regressor` · `RandomForestClassifier/Regressor` · `plot_tree` · `feature_importances_` · `max_depth`, `n_estimators`.

---
Carlos Enrique Mosquera Trujillo · cmosquerat@unal.edu.co  
**Machine Learning for Petroleum Engineers Using Python** · SLB Ecuador · UDLA · 2026

*Datos: FORCE 2020 (litología) y campo Volve, Equinor (operación) — datasets abiertos.*